# 01 — EDA: learner / cert / interaction corpus

**Project:** Micro-Cert Recommender (H07) · **Author:** Asad Kamran · MADS, University of Michigan · Dubai HR

Three parquet artefacts: 2,000 learners × 40 binary skill columns, 500 micro-certs with `skills_taught` text, and ~25,000 (learner, cert) interactions across enrol / complete / rate events. We profile them so the two-tower (collaborative + content) recommender in `03_model.ipynb` is grounded.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.insert(0, str(Path.cwd().parent / 'src'))
from microcert_rec.data import (
    PROCESSED, SKILL_VOCAB, THEME_LIST, ISSUERS,
    make_learners, make_certs, make_interactions, load_all, make_training_artifacts,
)

sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams['figure.dpi'] = 110

In [ ]:
if (PROCESSED / 'learners.parquet').exists():
    learners, certs, interactions = load_all()
else:
    learners, certs, interactions = make_training_artifacts()
learners.shape, certs.shape, interactions.shape

## 1. Per-learner skill prevalence

In [ ]:
skill_cols = [c for c in learners.columns if c.startswith('skill__')]
prevalence = learners[skill_cols].mean().sort_values()
fig, ax = plt.subplots(figsize=(10, 4))
prevalence.plot(kind='barh', ax=ax, color='#3a7ca5')
ax.set_xlabel('share of learners with skill')
ax.set_title('Per-skill learner prevalence')
plt.tight_layout(); plt.show()

## 2. Theme distribution per learner

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.4))
learners['primary_theme'].value_counts().plot(kind='bar', ax=ax, color='#9c6644')
ax.set_title('Primary theme per learner')
ax.tick_params(axis='x', rotation=30)
plt.tight_layout(); plt.show()

## 3. Cert catalog by issuer × theme

In [ ]:
issuer_theme = certs.groupby(['issuer', 'theme']).size().unstack(fill_value=0)
fig, ax = plt.subplots(figsize=(8, 4))
sns.heatmap(issuer_theme, cmap='YlGnBu', annot=True, fmt='d', ax=ax)
ax.set_title('Cert count: issuer × theme')
plt.tight_layout(); plt.show()

## 4. Cert hours + cost distributions

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.4))
sns.histplot(certs, x='hours', bins=30, ax=axes[0], color='#3a7ca5')
axes[0].set_title('Cert duration (hours)')
sns.histplot(certs, x='cost', bins=30, ax=axes[1], color='#4a7c59')
axes[1].set_title('Cert cost')
plt.tight_layout(); plt.show()

## 5. Interaction sparsity

In [ ]:
n_l, n_c = len(learners), len(certs)
density = len(interactions) / (n_l * n_c)
print(f'matrix density: {density:.4f} ({density*100:.2f}%)')
print(f'event mix: {interactions["event_type"].value_counts(normalize=True).round(3).to_dict()}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.4))
interactions.groupby('learner_id').size().plot(kind='hist', bins=30, ax=axes[0], color='#3a7ca5')
axes[0].set_title('# events per learner')
interactions.groupby('cert_id').size().plot(kind='hist', bins=30, ax=axes[1], color='#9c6644')
axes[1].set_title('# events per cert')
plt.tight_layout(); plt.show()

## 6. Popularity head — long tail check

In [ ]:
pop = interactions.groupby('cert_id').size().sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(8, 3.4))
ax.plot(np.arange(len(pop)), pop.values, color='#9c6644')
ax.set_xlabel('cert rank by popularity')
ax.set_ylabel('# events')
ax.set_title('Long-tail of cert popularity')
plt.tight_layout(); plt.show()

## 7. Hypotheses for `03_model.ipynb`

1. **Two-tower scoring will dominate either tower alone** — collaborative captures co-engagement; content captures cold-start.
2. **k=32 SVD factors** are sufficient at this density.
3. **Issuer-aware re-ranking** is a future enhancement — current scoring is issuer-agnostic; the cost column is informational only.